# CMiO SC-PaCMAP benchmark

## Study overview

**Research question.** Does SC-PaCMAP produce K-means partitions with a better balance of spatial contiguity and multivariate geochemical delineation than coordinate-aware UMAP, PaCMAP, LocalMAP, and SpaSNE?

This deterministic, full-population benchmark evaluates seven unsupervised embeddings. Deposit-group labels do not enter numeric preprocessing, embedding fitting, or K-means fitting. They define the evaluation classes and the label-informed $K=7$ reference; ARI measures evaluation-only agreement on the same full population.

The notebook first isolates the PaCMAP design choices in a focused six-method comparison, then evaluates all seven candidates using the same clustering and spatial-evaluation settings.

SC-PaCMAP benchmark runs use `SCPaCMAPConfig(max_target_share=1.0)`: no extra clipping below the natural bound of 1. Saved outputs have been cleared; rerun the notebook to regenerate results with this configuration. Any previously written numerical interpretations must be rechecked against the regenerated results.

In [ ]:
from __future__ import annotations

import hashlib
import importlib
import os
import platform
import sys
import time
from importlib.metadata import version
from pathlib import Path

os.environ["NUMBA_NUM_THREADS"] = "1"

import matplotlib.pyplot as plt
import numba
import numpy as np
import pacmap
import pandas as pd
import seaborn as sns
import umap
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler


def find_repository_root(start: Path) -> Path:
    ancestors = (start.resolve(), *start.resolve().parents)
    candidates = (*ancestors, *(path / "sc-pacmap-reproducibility" for path in ancestors))
    for candidate in candidates:
        if (
            (candidate / "code" / "sc_pacmap" / "sc_pacmap.py").is_file()
            and (candidate / "data" / "processed").is_dir()
        ):
            return candidate
    raise FileNotFoundError("Could not locate the sc-pacmap-reproducibility repository.")


ROOT = find_repository_root(Path.cwd())
SC_MODULE_DIR = ROOT / "code" / "sc_pacmap"
SPASNE_MODULE_DIR = ROOT / "code" / "spasne"
for module_dir in (SC_MODULE_DIR, SPASNE_MODULE_DIR):
    if str(module_dir) not in sys.path:
        sys.path.insert(0, str(module_dir))

from spasne_adapter import python_spasne

sc_pacmap_module = importlib.import_module("sc_pacmap")
sc_pacmap_module = importlib.reload(sc_pacmap_module)
PaCMAPSettings = sc_pacmap_module.PaCMAPSettings
SCPaCMAPConfig = sc_pacmap_module.SCPaCMAPConfig
sc_config = SCPaCMAPConfig(max_target_share=1.0)  # No extra clipping below 1.
run_legacy_sc_pacmap = sc_pacmap_module.run_legacy_sc_pacmap
run_sc_pacmap = sc_pacmap_module.run_sc_pacmap
verify_library_matches_copied_code = sc_pacmap_module.verify_library_matches_copied_code
verify_sc_disabled_matches_original = sc_pacmap_module.verify_sc_disabled_matches_original

if version("pacmap") != "0.9.1":
    raise RuntimeError("This benchmark requires pacmap==0.9.1.")
if numba.config.NUMBA_NUM_THREADS != 1:
    raise RuntimeError("Restart the kernel so NUMBA_NUM_THREADS=1 applies before import.")
numba.set_num_threads(1)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})

print(f"Python: {platform.python_version()}")
print(f"PaCMAP: {version('pacmap')}; UMAP: {version('umap-learn')}")
print(f"Repository: {ROOT}")

## Unified experiment configuration

All three dataset notebooks use the same deterministic workflow: seed 42, 10 neighbours, two embedding dimensions, Euclidean distance, PaCMAP-family `MN_ratio=0.5`, `FP_ratio=2.0`, learning rate 1.0, and optimization phases of $(100, 100, 250)$. The following cell declares these shared settings.

In [ ]:
# Unified deterministic benchmark parameters, matched to the prototype CMiO contract
SEED = 42
N_NEIGHBORS = 10
N_COMPONENTS = 2
KNOWN_CLASS_COUNT = 7
K_VALUES = range(2, 11)
SPASNE_ALPHA = 10.0
SPASNE_BETA = 5.0
SPASNE_EXAGGERATION = 4
SPASNE_PERPLEXITY = 50
SPASNE_THETA = 0.5
SPASNE_MAX_ITER = 1000
SPASNE_DIR = ROOT / "code" / "spasne" / "spasne_original"
XYZ_COLUMNS = ["x", "y", "z"]
LABEL_COLUMN = "LABEL_deposit_group"
FEATURE_COLUMNS = [
    "Y_PPM", "V_PPM", "LA_PPM", "U_PPM", "SR_PPM", "SN_PPM", "RB_PPM",
    "TH_PPM", "CS_PPM", "CE_PPM", "NB_PPM", "GA_PPM", "W_PPM", "HF_PPM",
    "TA_PPM", "ZR_PPM", "CR_PPM", "PR_PPM", "ND_PPM", "ER_PPM", "EU_PPM",
    "SM_PPM", "YB_PPM", "TB_PPM", "HO_PPM", "GD_PPM", "DY_PPM", "LU_PPM",
    "BA_PPM", "SIO2_WT_PERCENT", "TIO2_WT_PERCENT", "NA2O_WT_PERCENT",
    "K2O_WT_PERCENT", "AL2O3_WT_PERCENT", "TM_PPM",
]
METADATA_COLUMNS = [
    "sample_id", "group_id", *XYZ_COLUMNS, LABEL_COLUMN, "label_id",
]
EXPECTED_LABELS = {
    0: "Alkali-calcic",
    1: "IOCG",
    2: "Laterite",
    3: "Metasomatic iron",
    4: "Orogenic",
    5: "Phosphorite",
    6: "Sediment-hosted",
}

spasne_paths = {
    "upstream Python wrapper": SPASNE_DIR / "spasne.py",
    "project WSL bridge": SPASNE_DIR.parent / "spasne_wsl_bridge.py",
    "locally built Linux executable": SPASNE_DIR / "spasne",
}
missing_spasne_paths = [str(path) for path in spasne_paths.values() if not path.is_file()]
if missing_spasne_paths:
    raise RuntimeError(
        "SpaSNE is not built. Run 'make' in code/spasne/spasne_original "
        f"from Linux or WSL. Missing: {missing_spasne_paths}"
    )
spasne_configuration = pd.Series({
    "alpha": SPASNE_ALPHA, "beta": SPASNE_BETA,
    "exaggeration": SPASNE_EXAGGERATION, "perplexity": SPASNE_PERPLEXITY,
    "theta": SPASNE_THETA, "max_iter": SPASNE_MAX_ITER,
    "use_pca": True, "random_state": SEED,
}, name="SpaSNE published-default configuration")
display(spasne_configuration.to_frame())

print(
    f"Prototype-matched setup: seed={SEED}, neighbors={N_NEIGHBORS}, "
    f"dimensions={N_COMPONENTS}, K={min(K_VALUES)}-{max(K_VALUES)}."
)

## Data inspection

The CMiO input contains 1,551 rows, 35 geochemical variables, and raw metre-based XYZ coordinates. During preparation, negative source codes representing below-detection-limit assays were replaced by half their encoded limit. The data contract below verifies the complete population, feature schema, labels, and finite nonnegative numeric values before any embedding is fitted.

In [ ]:
DATA_PATH = ROOT / "data" / "processed" / "cmio_sc_pacmap_ready.csv"
data = pd.read_csv(DATA_PATH)

assert data.shape == (1551, 42)
assert data.columns.tolist() == METADATA_COLUMNS + FEATURE_COLUMNS
assert len(FEATURE_COLUMNS) == 35 and len(XYZ_COLUMNS) == 3
assert not data["sample_id"].duplicated().any()
assert not data[METADATA_COLUMNS + FEATURE_COLUMNS].isna().any().any()

features_raw = data[FEATURE_COLUMNS].to_numpy(dtype=np.float64)
coordinates_raw = data[XYZ_COLUMNS].to_numpy(dtype=np.float64)
labels_true = data["label_id"].to_numpy(dtype=np.int64)
label_names = data[LABEL_COLUMN].astype(str).to_numpy()

assert np.isfinite(features_raw).all() and np.isfinite(coordinates_raw).all()
assert not (features_raw < 0).any()
assert np.all(np.ptp(features_raw, axis=0) > 0)
assert set(np.unique(labels_true)) == set(EXPECTED_LABELS)
observed_labels = (
    data[["label_id", LABEL_COLUMN]]
    .drop_duplicates()
    .set_index("label_id")[LABEL_COLUMN]
    .to_dict()
)
assert observed_labels == EXPECTED_LABELS

# Match the prototype benchmark's feature precision and TSS calculation.
feature_scaler = StandardScaler()
coordinate_scaler = StandardScaler()
features_standardized = feature_scaler.fit_transform(features_raw).astype(np.float32)
coordinates_standardized = coordinate_scaler.fit_transform(coordinates_raw)
features_with_xyz = np.column_stack([features_standardized, coordinates_standardized])
feature_center = features_standardized.mean(axis=0, keepdims=True)
total_sum_squares = float(np.sum((features_standardized - feature_center) ** 2))
assert total_sum_squares > 0

contract = pd.Series({
    "rows": len(data),
    "geochemical features": len(FEATURE_COLUMNS),
    "coordinate columns": len(XYZ_COLUMNS),
    "known classes": data[LABEL_COLUMN].nunique(),
    "duplicate sample IDs": int(data["sample_id"].duplicated().sum()),
    "missing required values": int(data[METADATA_COLUMNS + FEATURE_COLUMNS].isna().sum().sum()),
    "negative retained values": int((data[FEATURE_COLUMNS] < 0).sum().sum()),
}, name="CMiO data contract")
display(contract.to_frame())
display(
    data.groupby(["label_id", LABEL_COLUMN], observed=True)
    .size()
    .rename("samples")
    .reset_index()
)

### Spatial support and evaluation-class balance

The following view maps the evaluation-only deposit groups in raw XYZ space and shows their sample counts. It is descriptive only: labels are not supplied to any embedding or K-means fitting step, and the same full population is used for evaluation rather than held-out validation.

In [ ]:
class_ids = list(EXPECTED_LABELS)
class_colors = dict(zip(class_ids, sns.color_palette("colorblind", len(class_ids))))

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
projections = [("x", "y"), ("x", "z"), ("y", "z")]
for axis, (horizontal, vertical) in zip(axes.flat[:3], projections):
    for class_id in class_ids:
        mask = labels_true == class_id
        axis.scatter(
            data.loc[mask, horizontal],
            data.loc[mask, vertical],
            s=16,
            alpha=0.7,
            color=class_colors[class_id],
            label=EXPECTED_LABELS[class_id],
            linewidth=0,
        )
    axis.set(title=f"{horizontal.upper()}{vertical.upper()} projection")
    axis.set_xlabel(horizontal.upper())
    axis.set_ylabel(vertical.upper())

counts = data["label_id"].value_counts().reindex(class_ids)
axes[1, 1].barh(
    [EXPECTED_LABELS[class_id] for class_id in class_ids],
    counts,
    color=[class_colors[class_id] for class_id in class_ids],
)
axes[1, 1].set(title="Class counts", xlabel="Samples", ylabel="")
axes[1, 1].invert_yaxis()

handles, legend_labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, legend_labels, loc="center left", bbox_to_anchor=(0.86, 0.5), frameon=False)
fig.suptitle("CMiO spatial support and deposit-group balance", fontsize=15)
plt.tight_layout(rect=(0, 0, 0.85, 0.96))
plt.show()

## Candidate embeddings

All candidates use two output dimensions and seed 42. UMAP and the PaCMAP family use 10 neighbours and Euclidean distance; PaCMAP-family candidates additionally use learning rate 1.0, `MN_ratio=0.5`, `FP_ratio=2.0`, and optimization phases of $(100, 100, 250)$. SpaSNE uses its published defaults without label-guided tuning: $\alpha=10$, $\beta=5$, perplexity 50, exaggeration 4, $\theta=0.5$, PCA preprocessing, and 1000 iterations.

- **PaCMAP** uses standardized geochemistry only [Wang et al. (2021), *JMLR* 22(201)].
- **PaCMAP + XYZ** appends separately standardized XYZ coordinates to the geochemical matrix.
- **SC-PaCMAP** is the legacy benchmark: it replaces part of PaCMAP's fixed-size feature-neighbour graph with PCA-Geary-ranked geographic neighbours.
- **SC-PaCMAP (full graph)** keeps the complete original PaCMAP feature graph, adds a separate weighted PCA-Geary graph, and transfers a data-adaptive share of nearest-neighbour attraction to the SC loss during Phase 3.
- **SpaSNE** receives standardized geochemistry and raw XYZ separately, combining feature, global-feature, and spatial probability terms [Tang et al., *Dimensionality reduction for visualizing spatially resolved profiling data using SpaSNE*].
- **UMAP + XYZ** uses standardized geochemistry plus XYZ [McInnes, Healy, and Melville (2018), arXiv:1802.03426].
- **LocalMAP + XYZ** uses standardized geochemistry plus XYZ [Wang et al. (2025), *AAAI* 39(20), DOI: 10.1609/aaai.v39i20.35436].

The focused full-graph comparison uses the same six curves as the supplied reference figure: UMAP + XYZ, PaCMAP + XYZ, LocalMAP + XYZ, feature-only PaCMAP, replacement-graph SC-PaCMAP, and full-graph SC-PaCMAP. SpaSNE remains in the broader benchmark.

### PaCMAP implementation check

Before fitting the candidates, SC-PaCMAP with its spatial rule disabled must reproduce the copied PaCMAP implementation exactly. The copied implementation is also compared with the installed `pacmap==0.9.1` library.

In [ ]:
settings = PaCMAPSettings(
    n_components=N_COMPONENTS,
    n_neighbors=N_NEIGHBORS,
    mn_ratio=0.5,
    fp_ratio=2.0,
    learning_rate=1.0,
    num_iters=(100, 100, 250),
    random_state=SEED,
    knn_backend="faiss",
    verbose=False,
)
audit_index = np.linspace(0, len(data) - 1, 200, dtype=int)
audit_features = np.ascontiguousarray(features_standardized[audit_index])
audit_coordinates = np.ascontiguousarray(coordinates_raw[audit_index])

sc_disabled_audit = verify_sc_disabled_matches_original(
    audit_features, audit_coordinates, settings
)
library_audit = verify_library_matches_copied_code(audit_features, settings)

required_equalities = [
    "neighbor_pairs_equal",
    "mid_near_pairs_equal",
    "further_pairs_equal",
    "embedding_equal",
]
assert all(sc_disabled_audit[key] for key in required_equalities)
assert sc_disabled_audit["maximum_embedding_error"] == 0.0
assert library_audit["status"] == "pass"
assert all(library_audit[key] for key in required_equalities)
assert library_audit["maximum_embedding_error"] == 0.0

audit_table = pd.DataFrame([
    {"audit": "SC disabled vs copied PaCMAP", **sc_disabled_audit},
    {"audit": "copied PaCMAP vs installed library", **library_audit},
])
display(audit_table)

### Fitted embeddings

Each candidate is fitted to the full population. The output table records input pathways and runtime; the following diagnostic view colors the embeddings by the evaluation-only deposit group for inspection.

In [ ]:
METHOD_ORDER = [
    "UMAP + XYZ", "PaCMAP + XYZ", "LocalMAP + XYZ",
    "PaCMAP feature-only", "SC-PaCMAP (neighbour replacement graph)",
    "SC-PaCMAP (full graph)", "SpaSNE",
]
FOCUSED_METHOD_ORDER = [
    "UMAP + XYZ", "PaCMAP + XYZ", "LocalMAP + XYZ",
    "PaCMAP feature-only", "SC-PaCMAP (neighbour replacement graph)",
    "SC-PaCMAP (full graph)",
]
embeddings = {}
runtimes = {}


def store_embedding(method: str, embedding: np.ndarray, started_at: float) -> None:
    embedding = np.asarray(embedding, dtype=np.float64)
    assert embedding.shape == (len(data), N_COMPONENTS)
    assert np.isfinite(embedding).all()
    embeddings[method] = embedding
    runtimes[method] = time.perf_counter() - started_at


def fit_localmap_in_clean_process(x_input: np.ndarray, n_neighbors: int) -> np.ndarray:
    import subprocess

    worker_code = r'''
import os
os.environ["NUMBA_NUM_THREADS"] = "1"
import importlib
import sys
import numba
import numpy as np
import pacmap

@numba.njit
def seed_numba_rng(random_state):
    np.random.seed(random_state)

pacmap_core = importlib.import_module("pacmap.pacmap")
original_resample = pacmap_core.sample_FP_pair_nearby
resample_call_index = 0

def deterministic_resample(X, pair_neighbors, old_pair_FP, Y, low_dist_thres):
    global resample_call_index
    seed_numba_rng(42 + resample_call_index)
    resample_call_index += 1
    return original_resample(X, pair_neighbors, old_pair_FP, Y, low_dist_thres)

pacmap_core.sample_FP_pair_nearby = deterministic_resample
rows, columns, n_neighbors = map(int, sys.argv[1:4])
x_input = np.frombuffer(sys.stdin.buffer.read(), dtype=np.float64).reshape(rows, columns)
seed_numba_rng(42)
model = pacmap.LocalMAP(
    n_components=2, n_neighbors=n_neighbors, MN_ratio=0.5, FP_ratio=2.0,
    distance="euclidean", lr=1.0, num_iters=(100, 100, 250),
    apply_pca=True, random_state=42, save_tree=False, low_dist_thres=10,
    knn_backend="faiss",
)
embedding = model.fit_transform(x_input, init="pca", save_pairs=False)
sys.stdout.buffer.write(np.asarray(embedding, dtype=np.float32).tobytes())
'''
    x_input = np.ascontiguousarray(x_input, dtype=np.float64)
    completed = subprocess.run(
        [sys.executable, "-c", worker_code, *map(str, x_input.shape), str(n_neighbors)],
        input=x_input.tobytes(), capture_output=True, check=False,
    )
    if completed.returncode != 0:
        raise RuntimeError(completed.stderr.decode("utf-8", errors="replace"))
    expected_bytes = len(x_input) * N_COMPONENTS * np.dtype(np.float32).itemsize
    if len(completed.stdout) != expected_bytes:
        raise RuntimeError("LocalMAP worker returned an unexpected embedding size.")
    return np.frombuffer(completed.stdout, dtype=np.float32).reshape(
        len(x_input), N_COMPONENTS
    ).copy()


started_at = time.perf_counter()
umap_model = umap.UMAP(
    n_neighbors=N_NEIGHBORS, n_components=N_COMPONENTS, metric="euclidean",
    learning_rate=1.0, random_state=SEED, n_jobs=1,
)
store_embedding("UMAP + XYZ", umap_model.fit_transform(features_with_xyz), started_at)

pacmap_parameters = {
    "n_components": N_COMPONENTS, "n_neighbors": N_NEIGHBORS,
    "MN_ratio": 0.5, "FP_ratio": 2.0, "distance": "euclidean",
    "lr": 1.0, "num_iters": (100, 100, 250), "apply_pca": True,
    "random_state": SEED, "save_tree": False, "knn_backend": "faiss",
}

started_at = time.perf_counter()
pacmap_xyz_model = pacmap.PaCMAP(**pacmap_parameters)
store_embedding(
    "PaCMAP + XYZ",
    pacmap_xyz_model.fit_transform(features_with_xyz, init="pca", save_pairs=False),
    started_at,
)

started_at = time.perf_counter()
store_embedding(
    "LocalMAP + XYZ",
    fit_localmap_in_clean_process(features_with_xyz, N_NEIGHBORS),
    started_at,
)

started_at = time.perf_counter()
pacmap_feature_model = pacmap.PaCMAP(**pacmap_parameters)
store_embedding(
    "PaCMAP feature-only",
    pacmap_feature_model.fit_transform(features_standardized, init="pca", save_pairs=False),
    started_at,
)

started_at = time.perf_counter()
replacement_sc_result, replacement_sc_diagnostics = run_legacy_sc_pacmap(
    features_standardized, coordinates_raw, settings
)
store_embedding("SC-PaCMAP (neighbour replacement graph)", replacement_sc_result.embedding, started_at)
assert replacement_sc_diagnostics.k_spatial + replacement_sc_diagnostics.k_attribute == N_NEIGHBORS

started_at = time.perf_counter()
sc_result, sc_diagnostics = run_sc_pacmap(
    features_standardized, coordinates_raw, settings, config=sc_config
)
store_embedding("SC-PaCMAP (full graph)", sc_result.embedding, started_at)
assert sc_diagnostics is not None
assert sc_diagnostics.k_attribute == N_NEIGHBORS
assert sc_result.pair_neighbors.shape == (len(data) * N_NEIGHBORS, 2)
assert sc_diagnostics.pair_sc.shape == (len(data) * sc_diagnostics.k_spatial, 2)
assert 0.0 <= sc_diagnostics.spatial_strength_lambda <= 1.0
assert 0.0 <= sc_diagnostics.automatic_target_share <= 1.0

started_at = time.perf_counter()
spasne_embedding = python_spasne(
    features_standardized,
    coordinates_raw,
    random_state=SEED,
    alpha=SPASNE_ALPHA,
    beta=SPASNE_BETA,
    exaggeration=SPASNE_EXAGGERATION,
    perplexity=SPASNE_PERPLEXITY,
    theta=SPASNE_THETA,
    max_iter=SPASNE_MAX_ITER,
    use_pca=True,
    spasne_dir=SPASNE_DIR,
)
store_embedding("SpaSNE", spasne_embedding, started_at)

embedding_cluster_inputs = embeddings
runtime_table = pd.DataFrame({
    "method": METHOD_ORDER,
    "seconds": [runtimes[method] for method in METHOD_ORDER],
    "geochemical input columns": [35] * len(METHOD_ORDER),
    "coordinate pathway": [
        "standardized XYZ columns", "standardized XYZ columns",
        "standardized XYZ columns", "none",
        "raw XYZ neighbour replacement graph", "raw XYZ separate Phase-3 SC graph",
        "raw XYZ spatial objective",
    ],
})
display(runtime_table.round({"seconds": 3}))

### Embedding diagnostics and visual inspection

Embedding fingerprints, SC-PaCMAP diagnostics, and the evaluation-label embedding view document the fitted candidates before evaluation.

In [ ]:
embedding_fingerprint_table = pd.DataFrame({
    "method": METHOD_ORDER,
    "embedding SHA-256": [
        hashlib.sha256(
            np.ascontiguousarray(embeddings[method]).tobytes()
        ).hexdigest()
        for method in METHOD_ORDER
    ],
})
display(embedding_fingerprint_table)

diagnostic_summary = pd.Series({
    "legacy spatial-strength lambda": replacement_sc_diagnostics.spatial_strength_lambda,
    "legacy replaced spatial slots": replacement_sc_diagnostics.k_spatial,
    "legacy retained attribute slots": replacement_sc_diagnostics.k_attribute,
    "enhanced spatial-strength lambda": sc_diagnostics.spatial_strength_lambda,
    "enhanced retained feature slots": sc_diagnostics.k_attribute,
    "enhanced additional SC slots": sc_diagnostics.k_spatial,
    "enhanced automatic target share": sc_diagnostics.automatic_target_share,
    "enhanced mean realized active share": sc_diagnostics.mean_realized_sc_share_active,
    "enhanced directed overlap fraction": sc_diagnostics.overlap_fraction,
    "enhanced mean SC weight after overlap": sc_diagnostics.mean_sc_weight_after_overlap,
    "Geary C minimum": np.min(sc_diagnostics.geary_c),
    "Geary C median": np.median(sc_diagnostics.geary_c),
    "Geary C maximum": np.max(sc_diagnostics.geary_c),
}, name="SC-PaCMAP graph and Phase-3 diagnostics")
display(diagnostic_summary.to_frame().round(4))

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for axis, method in zip(axes.flat, METHOD_ORDER):
    embedding = embeddings[method]
    for class_id in class_ids:
        mask = labels_true == class_id
        axis.scatter(
            embedding[mask, 0],
            embedding[mask, 1],
            s=14,
            alpha=0.68,
            color=class_colors[class_id],
            label=EXPECTED_LABELS[class_id],
            linewidth=0,
        )
    axis.set_title(method)
    axis.set_xlabel("Embedding 1")
    axis.set_ylabel("Embedding 2")
    axis.grid(alpha=0.25)
for axis in axes.flat[len(METHOD_ORDER):]:
    axis.set_visible(False)
handles, legend_labels = axes.flat[0].get_legend_handles_labels()
fig.legend(
    handles,
    legend_labels,
    loc="lower center",
    ncol=4,
    frameon=False,
    bbox_to_anchor=(0.5, 0.01),
)
fig.suptitle("CMiO embeddings colored by evaluation-only deposit group", fontsize=15)
plt.tight_layout(rect=(0, 0.08, 1, 0.96))
plt.show()

## Evaluation measures

K-means receives each raw 2-D embedding directly, without further standardization, normalization, rotation, or alignment. Spatial continuity uses each sample's 10 nearest non-self raw-XYZ neighbours:

$$\frac{H}{N} = -\frac{1}{N}\sum_{i=1}^{N}\sum_j p_{ij}\log p_{ij}.$$

Geochemical delineation uses the standardized feature space:

$$\frac{\mathrm{WCSS}}{\mathrm{TSS}} = \frac{\sum_k\sum_{i \in k}\lVert x_i-\bar{x}_k\rVert^2}{\sum_i\lVert x_i-\bar{x}\rVert^2}.$$

Lower entropy and WCSS/TSS are preferable; ARI is an evaluation-only deposit-group reference computed on the same full population.

## Result board

For each embedding and each $K=2,\ldots,10$, deterministic K-means uses 20 initializations. The board reports spatial continuity, geochemical delineation, and evaluation-only deposit-group agreement for the complete candidate set.

In [ ]:
# Prototype-matched evaluation: raw 2-D embeddings enter K-means unchanged.
spatial_model = NearestNeighbors(metric="euclidean", n_jobs=1).fit(coordinates_raw)
spatial_neighbors = spatial_model.kneighbors(
    coordinates_raw, n_neighbors=N_NEIGHBORS + 1, return_distance=False,
)[:, 1:]
assert spatial_neighbors.shape == (len(data), N_NEIGHBORS)

spatial_context = pd.Series({
    "raw-XYZ neighbors per sample": N_NEIGHBORS,
    "focal sample excluded": True,
    "K-means embedding input": "raw 2-D embedding (not standardized)",
}, name="prototype-matched evaluation context")
display(spatial_context.to_frame())


def total_windowed_entropy(cluster_labels: np.ndarray) -> float:
    total = 0.0
    for neighbors in spatial_neighbors:
        counts = np.bincount(cluster_labels[neighbors])
        probabilities = counts[counts > 0] / N_NEIGHBORS
        total -= float(np.sum(probabilities * np.log(probabilities)))
    return total


def feature_space_wcss(cluster_labels: np.ndarray, n_clusters: int) -> float:
    centroids = np.vstack([
        features_standardized[cluster_labels == cluster_id].mean(axis=0)
        for cluster_id in range(n_clusters)
    ])
    residuals = features_standardized - centroids[cluster_labels]
    return float(np.sum(residuals * residuals))


result_rows = []
for method in METHOD_ORDER:
    raw_embedding = embeddings[method]
    assert raw_embedding.shape == (len(data), N_COMPONENTS)
    for k in K_VALUES:
        cluster_labels = KMeans(
            n_clusters=k,
            init="k-means++",
            n_init=20,
            random_state=1000 + k,
        ).fit_predict(raw_embedding)
        result_rows.append({
            "method": method,
            "K": k,
            "H_per_sample": total_windowed_entropy(cluster_labels) / len(data),
            "WCSS_TSS": feature_space_wcss(cluster_labels, k) / total_sum_squares,
            "ARI": adjusted_rand_score(labels_true, cluster_labels),
        })

results = pd.DataFrame(result_rows)
results["method"] = pd.Categorical(
    results["method"], categories=METHOD_ORDER, ordered=True
)
results = results.sort_values(["method", "K"]).reset_index(drop=True)

assert results.columns.tolist() == ["method", "K", "H_per_sample", "WCSS_TSS", "ARI"]
assert len(results) == len(METHOD_ORDER) * len(list(K_VALUES))
assert not results.duplicated(["method", "K"]).any()
assert results.groupby("method", observed=True)["K"].apply(
    lambda values: list(values) == list(K_VALUES)
).all()
assert np.isfinite(results[["H_per_sample", "WCSS_TSS", "ARI"]]).all().all()
assert results["H_per_sample"].ge(0).all() and results["WCSS_TSS"].gt(0).all()
assert results["ARI"].between(-1, 1).all()

display(results.round({"H_per_sample": 4, "WCSS_TSS": 4, "ARI": 4}))

In [ ]:
known_k_results = results.loc[
    results["K"] == KNOWN_CLASS_COUNT,
    ["method", "H_per_sample", "WCSS_TSS", "ARI"],
].copy()
display(known_k_results.round({"H_per_sample": 4, "WCSS_TSS": 4, "ARI": 4}))

fingerprint_table = results.sort_values(["method", "K"]).copy()
fingerprint_table["method"] = fingerprint_table["method"].astype(str)
fingerprint_payload = fingerprint_table.to_csv(
    index=False, float_format="%.12g", lineterminator="\n"
).encode("utf-8")
metric_fingerprint = hashlib.sha256(fingerprint_payload).hexdigest()
print(f"Metric-table SHA-256: {metric_fingerprint}")

In [ ]:
# K=7 PaCMAP-family clusters mapped back to original XY
ABLATION_METHOD_ORDER = [
    "PaCMAP feature-only",
    "PaCMAP + XYZ",
    "SC-PaCMAP (neighbour replacement graph)",
    "SC-PaCMAP (full graph)",
]
ablation_cluster_labels = {
    method: KMeans(
        n_clusters=KNOWN_CLASS_COUNT,
        init="k-means++",
        n_init=20,
        random_state=1000 + KNOWN_CLASS_COUNT,
    ).fit_predict(embedding_cluster_inputs[method])
    for method in ABLATION_METHOD_ORDER
}

fig, axes = plt.subplots(2, 2, figsize=(12, 9), constrained_layout=True)
for axis, method in zip(axes.flat, ABLATION_METHOD_ORDER):
    axis.scatter(
        coordinates_raw[:, 0],
        coordinates_raw[:, 1],
        c=ablation_cluster_labels[method],
        cmap="tab10",
        s=10,
        alpha=0.72,
        linewidth=0,
    )
    axis.set_title(method)
    axis.set_xlabel("X (m)")
    axis.set_ylabel("Y (m)")
    axis.ticklabel_format(style="plain", useOffset=False)
fig.suptitle(f"K-means partitions at K={KNOWN_CLASS_COUNT} in original XY", fontsize=14)
plt.show()

## PaCMAP-focused ablation

The focused comparison maps feature-only PaCMAP, PaCMAP + XYZ, and SC-PaCMAP back to original XY at $K=7$, then compares their metric trajectories and spatial-geochemical trade-off.

In [ ]:
# PaCMAP-family ablation across K
ABLATION_METHOD_ORDER = [
    "PaCMAP feature-only",
    "PaCMAP + XYZ",
    "SC-PaCMAP (neighbour replacement graph)",
    "SC-PaCMAP (full graph)",
]
method_colors = {
    "UMAP + XYZ": "#4C78A8",
    "PaCMAP + XYZ": "#F28E2B",
    "LocalMAP + XYZ": "#59A14F",
    "PaCMAP feature-only": "#B07AA1",
    "SC-PaCMAP (neighbour replacement graph)": "#E15759",
    "SC-PaCMAP (full graph)": "#2457D6",
    "SpaSNE": "#79706E",
}
method_markers = {
    "UMAP + XYZ": "o",
    "PaCMAP + XYZ": "s",
    "LocalMAP + XYZ": "^",
    "PaCMAP feature-only": "D",
    "SC-PaCMAP (neighbour replacement graph)": "P",
    "SC-PaCMAP (full graph)": "*",
    "SpaSNE": "X",
}

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), constrained_layout=True)
metric_specs = [
    ("H_per_sample", "Mean windowed entropy, H/N", "Spatial continuity (lower is better)"),
    ("WCSS_TSS", "WCSS / TSS", "Geochemical delineation (lower is better)"),
    ("ARI", "Adjusted Rand index", "Evaluation-only deposit-group agreement (higher is better)"),
]
for axis, (metric, ylabel, title) in zip(axes, metric_specs):
    for method in ABLATION_METHOD_ORDER:
        method_results = results.loc[results["method"] == method].sort_values("K")
        axis.plot(
            method_results["K"],
            method_results[metric],
            color=method_colors[method],
            marker=method_markers[method],
            linewidth=2,
            markersize=6,
            label=method,
        )
    axis.axvline(KNOWN_CLASS_COUNT, color="black", linewidth=1, linestyle="--", alpha=0.6)
    if metric == "ARI":
        axis.axhline(0, color="black", linewidth=1, linestyle=":")
    axis.set(
        xlabel="K-means clusters (K)",
        ylabel=ylabel,
        title=title,
    )
    axis.set_xticks(list(K_VALUES))
    axis.grid(alpha=0.25)
handles, legend_labels = axes[0].get_legend_handles_labels()
fig.legend(handles, legend_labels, loc="outside lower center", ncol=4, frameon=False)
fig.suptitle("PaCMAP-family ablation", fontsize=15)
plt.show()

### Automatic-share focused trade-off

The six curves match the supplied reference comparison. Marker size increases with $K$ and each black-edged star identifies $K=7$. Movement toward the lower-left combines more contiguous spatial partitions with tighter geochemical clusters.

In [ ]:
marker_sizes = {k: 45 + 22 * (k - min(K_VALUES)) for k in K_VALUES}

_, axis = plt.subplots(figsize=(11, 6.8))
for method in FOCUSED_METHOD_ORDER:
    method_results = results.loc[results["method"] == method].sort_values("K")
    axis.plot(
        method_results["H_per_sample"],
        method_results["WCSS_TSS"],
        color=method_colors[method],
        linewidth=1.5,
        alpha=0.72,
    )
    axis.scatter(
        method_results["H_per_sample"],
        method_results["WCSS_TSS"],
        s=[marker_sizes[k] for k in method_results["K"]],
        marker=method_markers[method],
        color=method_colors[method],
        edgecolor="white",
        linewidth=0.7,
        alpha=0.92,
        label=method,
        zorder=3,
    )
    reference_point = method_results.loc[method_results["K"] == KNOWN_CLASS_COUNT]
    axis.scatter(
        reference_point["H_per_sample"],
        reference_point["WCSS_TSS"],
        s=marker_sizes[KNOWN_CLASS_COUNT] + 55,
        marker="*",
        facecolor=method_colors[method],
        edgecolor="black",
        linewidth=1.4,
        zorder=5,
    )

axis.set(
    xlabel="Mean windowed entropy H/N (lower is better)",
    ylabel="WCSS/TSS in standardized feature space (lower is better)",
    title="CMiO spatial-contiguity and feature-delineation trade-off",
)
axis.annotate(
    "Enhanced SC (automatic)",
    xy=(
        results.loc[
            (results["method"] == "SC-PaCMAP (full graph)")
            & (results["K"] == KNOWN_CLASS_COUNT),
            "H_per_sample",
        ].iloc[0],
        results.loc[
            (results["method"] == "SC-PaCMAP (full graph)")
            & (results["K"] == KNOWN_CLASS_COUNT),
            "WCSS_TSS",
        ].iloc[0],
    ),
    xytext=(-105, -18),
    textcoords="offset points",
    color=method_colors["SC-PaCMAP (full graph)"],
    fontsize=9,
    fontweight="bold",
    arrowprops={"arrowstyle": "->", "color": method_colors["SC-PaCMAP (full graph)"]},
)
axis.grid(alpha=0.3)
axis.legend(frameon=False, ncol=2, fontsize=8, loc="upper right")
axis.text(
    0.98, 0.02,
    f"Marker size increases with K; stars show K={KNOWN_CLASS_COUNT}",
    transform=axis.transAxes, ha="right", va="bottom", fontsize=8, color="#53657A",
)
plt.tight_layout()
plt.show()

## Full seven-method benchmark

This section retains SpaSNE in addition to the six-method full-graph comparison. The input pathways remain explicit: the three `+ XYZ` candidates use standardized coordinates as ordinary input columns; feature-only PaCMAP receives no coordinates; replacement-graph SC-PaCMAP uses raw XYZ to replace part of its fixed-size NN graph; full-graph SC-PaCMAP retains the full feature graph and uses raw XYZ only for a separate Phase-3 SC loss; SpaSNE receives raw XYZ separately through its spatial objective.

In [ ]:
# All seven candidates across K
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), constrained_layout=True)
metric_specs = [
    ("H_per_sample", "Mean windowed entropy, H/N", "Spatial continuity (lower is better)"),
    ("WCSS_TSS", "WCSS / TSS", "Geochemical delineation (lower is better)"),
    ("ARI", "Adjusted Rand index", "Evaluation-only deposit-group agreement (higher is better)"),
]
for axis, (metric, ylabel, title) in zip(axes, metric_specs):
    for method in METHOD_ORDER:
        method_results = results.loc[results["method"] == method].sort_values("K")
        axis.plot(
            method_results["K"],
            method_results[metric],
            color=method_colors[method],
            marker=method_markers[method],
            linewidth=2,
            markersize=5.5,
            label=method,
        )
    axis.axvline(KNOWN_CLASS_COUNT, color="black", linewidth=1, linestyle="--", alpha=0.6)
    if metric == "ARI":
        axis.axhline(0, color="black", linewidth=1, linestyle=":")
    axis.set(
        xlabel="K-means clusters (K)",
        ylabel=ylabel,
        title=title,
    )
    axis.set_xticks(list(K_VALUES))
    axis.grid(alpha=0.25)
handles, legend_labels = axes[0].get_legend_handles_labels()
fig.legend(handles, legend_labels, loc="outside lower center", ncol=4, frameon=False)
fig.suptitle("Full seven-method benchmark", fontsize=15)
plt.show()

### Full benchmark trade-off

Each point is a method-by-$K$ solution; marker size increases with $K$, and the black ring identifies the seven-class reference. The lower-left direction is preferable jointly, but no single scalar is used to collapse the two evaluation objectives.

In [ ]:
# All-candidate spatial-geochemical trade-off
_, axis = plt.subplots(figsize=(10.5, 6.8))
for method in METHOD_ORDER:
    method_results = results.loc[results["method"] == method].sort_values("K")
    axis.plot(
        method_results["H_per_sample"],
        method_results["WCSS_TSS"],
        color=method_colors[method],
        linewidth=1.4,
        alpha=0.55,
    )
    axis.scatter(
        method_results["H_per_sample"],
        method_results["WCSS_TSS"],
        s=[marker_sizes[k] for k in method_results["K"]],
        marker=method_markers[method],
        color=method_colors[method],
        edgecolor="white",
        linewidth=0.7,
        alpha=0.9,
        label=method,
        zorder=3,
    )
    known_point = method_results.loc[method_results["K"] == KNOWN_CLASS_COUNT]
    axis.scatter(
        known_point["H_per_sample"],
        known_point["WCSS_TSS"],
        s=marker_sizes[KNOWN_CLASS_COUNT] + 35,
        marker=method_markers[method],
        facecolor="none",
        edgecolor="black",
        linewidth=1.8,
        zorder=4,
    )
axis.set(
    xlabel="Mean windowed entropy H/N (lower is better)",
    ylabel="WCSS/TSS in standardized feature space (lower is better)",
    title="Full seven-method spatial-geochemical trade-off",
)
axis.annotate(
    "Jointly lower",
    xy=(0.035, 0.04),
    xytext=(0.18, 0.17),
    xycoords="axes fraction",
    arrowprops={"arrowstyle": "->", "color": "black"},
)
axis.grid(alpha=0.3)
axis.legend(title="Method", frameon=False, loc="upper left", bbox_to_anchor=(1.01, 1.0))
plt.tight_layout(rect=(0, 0, 0.8, 1))
plt.show()

## Summary at the seven-class reference

The summary below identifies the best observed value for each metric at the label-informed $K=7$ reference and quantifies the SC-PaCMAP change relative to feature-only PaCMAP. ARI is an in-sample, evaluation-only agreement measure, not held-out or independent validation.

In [ ]:
best_entropy = known_k_results.loc[known_k_results["H_per_sample"].idxmin()]
best_wcss = known_k_results.loc[known_k_results["WCSS_TSS"].idxmin()]
best_ari = known_k_results.loc[known_k_results["ARI"].idxmax()]
overall_ari_best = results.loc[results["ARI"].idxmax()]
full_graph_known = known_k_results.loc[
    known_k_results["method"] == "SC-PaCMAP (full graph)"
] .iloc[0]
replacement_graph_known = known_k_results.loc[
    known_k_results["method"] == "SC-PaCMAP (neighbour replacement graph)"
] .iloc[0]
feature_known = known_k_results.loc[
    known_k_results["method"] == "PaCMAP feature-only"
] .iloc[0]
entropy_reduction = 100 * (
    feature_known["H_per_sample"] - full_graph_known["H_per_sample"]
) / feature_known["H_per_sample"]
wcss_change = 100 * (
    full_graph_known["WCSS_TSS"] - feature_known["WCSS_TSS"]
) / feature_known["WCSS_TSS"]
replacement_graph_wcss_recovery = 100 * (
    replacement_graph_known["WCSS_TSS"] - full_graph_known["WCSS_TSS"]
) / replacement_graph_known["WCSS_TSS"]

print(
    f"At K={KNOWN_CLASS_COUNT}, {best_entropy['method']} has the lowest entropy "
    f"H/N ({best_entropy['H_per_sample']:.4f}), {best_wcss['method']} has the lowest "
    f"WCSS/TSS ({best_wcss['WCSS_TSS']:.4f}), and {best_ari['method']} has the highest "
    f"ARI ({best_ari['ARI']:.4f})."
)
print(
    f"Relative to feature-only PaCMAP at K={KNOWN_CLASS_COUNT}, automatic-share "
    f"full-graph SC-PaCMAP reduces entropy by {entropy_reduction:.1f}% while changing "
    f"WCSS by {wcss_change:+.1f}%."
)
print(
    f"Relative to replacement-graph SC-PaCMAP, the full-graph method lowers WCSS/TSS by "
    f"{replacement_graph_wcss_recovery:.1f}%."
)
print(
    f"The largest ARI in the complete sweep is {overall_ari_best['ARI']:.4f} from "
    f"{overall_ari_best['method']} at K={int(overall_ari_best['K'])}."
)

In [ ]:
####################### PUBLICATION NEEDED STATISTICS #########################################

## All-K stability bands

Every method is refit with algorithm seeds 11, 42, and 97 and evaluated across $K=2,\ldots,10$. The solid line is the across-seed median and the shaded band is the min-max range at each $K$. These bands describe optimizer sensitivity on this one fixed CMiO dataset; they are **not confidence intervals**, and no population-level significance test is applied.

In [ ]:
ALGORITHM_SEEDS = (11, 42, 97)
STABILITY_METHOD_ORDER = list(METHOD_ORDER)


def fit_localmap_for_seed(x_input: np.ndarray, algorithm_seed: int) -> np.ndarray:
    import subprocess

    worker_code = r'''
import importlib
import os
import sys
os.environ["NUMBA_NUM_THREADS"] = "1"
import numba
import numpy as np
import pacmap

@numba.njit
def seed_numba_rng(random_state):
    np.random.seed(random_state)

rows, columns, n_neighbors, algorithm_seed = map(int, sys.argv[1:5])
pacmap_core = importlib.import_module("pacmap.pacmap")
original_resample = pacmap_core.sample_FP_pair_nearby
resample_call_index = 0

def deterministic_resample(X, pair_neighbors, old_pair_FP, Y, low_dist_thres):
    global resample_call_index
    seed_numba_rng(algorithm_seed + resample_call_index)
    resample_call_index += 1
    return original_resample(X, pair_neighbors, old_pair_FP, Y, low_dist_thres)

pacmap_core.sample_FP_pair_nearby = deterministic_resample
x_input = np.frombuffer(sys.stdin.buffer.read(), dtype=np.float64).reshape(rows, columns)
seed_numba_rng(algorithm_seed)
model = pacmap.LocalMAP(
    n_components=2, n_neighbors=n_neighbors, MN_ratio=0.5, FP_ratio=2.0,
    distance="euclidean", lr=1.0, num_iters=(100, 100, 250),
    apply_pca=True, random_state=algorithm_seed, save_tree=False,
    low_dist_thres=10, knn_backend="faiss",
)
embedding = model.fit_transform(x_input, init="pca", save_pairs=False)
sys.stdout.buffer.write(np.asarray(embedding, dtype=np.float32).tobytes())
'''
    x_input = np.ascontiguousarray(x_input, dtype=np.float64)
    arguments = [*map(str, x_input.shape), str(N_NEIGHBORS), str(algorithm_seed)]
    completed = subprocess.run(
        [sys.executable, "-c", worker_code, *arguments],
        input=x_input.tobytes(), capture_output=True, check=False,
    )
    if completed.returncode != 0:
        raise RuntimeError(completed.stderr.decode("utf-8", errors="replace"))
    return np.frombuffer(completed.stdout, dtype=np.float32).reshape(
        len(x_input), N_COMPONENTS
    ).copy()


def fit_embeddings_for_seed(algorithm_seed: int) -> dict[str, np.ndarray]:
    if algorithm_seed == SEED:
        return embeddings

    seed_settings = PaCMAPSettings(
        n_components=N_COMPONENTS, n_neighbors=N_NEIGHBORS,
        mn_ratio=0.5, fp_ratio=2.0, learning_rate=1.0,
        num_iters=(100, 100, 250), random_state=algorithm_seed,
        knn_backend="faiss", verbose=False,
    )
    seed_pacmap_parameters = {**pacmap_parameters, "random_state": algorithm_seed}
    seed_embeddings = {
        "UMAP + XYZ": umap.UMAP(
            n_neighbors=N_NEIGHBORS, n_components=N_COMPONENTS,
            metric="euclidean", learning_rate=1.0,
            random_state=algorithm_seed, n_jobs=1,
        ).fit_transform(features_with_xyz),
        "PaCMAP + XYZ": pacmap.PaCMAP(**seed_pacmap_parameters).fit_transform(
            features_with_xyz, init="pca", save_pairs=False
        ),
        "LocalMAP + XYZ": fit_localmap_for_seed(features_with_xyz, algorithm_seed),
        "PaCMAP feature-only": pacmap.PaCMAP(**seed_pacmap_parameters).fit_transform(
            features_standardized, init="pca", save_pairs=False
        ),
    }
    replacement_result, _ = run_legacy_sc_pacmap(
        features_standardized, coordinates_raw, seed_settings
    )
    full_graph_result, _ = run_sc_pacmap(
        features_standardized, coordinates_raw, seed_settings, config=sc_config
    )
    seed_embeddings["SC-PaCMAP (neighbour replacement graph)"] = replacement_result.embedding
    seed_embeddings["SC-PaCMAP (full graph)"] = full_graph_result.embedding
    seed_embeddings["SpaSNE"] = python_spasne(
        features_standardized, coordinates_raw, random_state=algorithm_seed,
        alpha=SPASNE_ALPHA, beta=SPASNE_BETA,
        exaggeration=SPASNE_EXAGGERATION, perplexity=SPASNE_PERPLEXITY,
        theta=SPASNE_THETA, max_iter=SPASNE_MAX_ITER, use_pca=True,
        spasne_dir=SPASNE_DIR,
    )
    return seed_embeddings


def evaluate_partition(
    method: str, algorithm_seed: int, k: int, cluster_labels: np.ndarray
) -> dict:
    return {
        "method": method,
        "algorithm_seed": algorithm_seed,
        "K": k,
        "H_per_sample": total_windowed_entropy(cluster_labels) / len(data),
        "WCSS_TSS": feature_space_wcss(cluster_labels, k) / total_sum_squares,
        "ARI": adjusted_rand_score(labels_true, cluster_labels),
    }


stability_rows = []
for algorithm_seed in ALGORITHM_SEEDS:
    seed_embeddings = fit_embeddings_for_seed(algorithm_seed)
    for method in METHOD_ORDER:
        for k in K_VALUES:
            labels = KMeans(
                n_clusters=k, init="k-means++", n_init=20, random_state=1000 + k,
            ).fit_predict(seed_embeddings[method])
            stability_rows.append(evaluate_partition(method, algorithm_seed, k, labels))

stability_results = pd.DataFrame(stability_rows)
expected_rows = len(K_VALUES) * len(METHOD_ORDER) * len(ALGORITHM_SEEDS)
assert len(stability_results) == expected_rows == 189
assert not stability_results.duplicated(["method", "algorithm_seed", "K"]).any()
assert np.isfinite(stability_results[["H_per_sample", "WCSS_TSS", "ARI"]]).all().all()

output_path = ROOT / "results" / "cmio_algorithm_stability.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
stability_results.to_csv(output_path, index=False)
display(stability_results.round(4))
print(f"Saved {len(stability_results)} rows to {output_path.relative_to(ROOT)}")

In [ ]:
stability_results = pd.read_csv(ROOT / "results" / "cmio_algorithm_stability.csv")
method_styles = {
    "UMAP + XYZ": ("#D55E00", "o", "-"),
    "PaCMAP + XYZ": ("#56B4E9", "^", "-"),
    "LocalMAP + XYZ": ("#009E73", "v", "-"),
    "PaCMAP feature-only": ("#7A5195", "P", "-"),
    "SC-PaCMAP (neighbour replacement graph)": ("#E15759", "h", "-"),
    "SC-PaCMAP (full graph)": ("#E60049", "*", "-"),
    "SpaSNE": ("#D4A900", "X", "-"),
}
metrics = [
    ("H_per_sample", "Mean windowed entropy H/N", "lower is better"),
    ("WCSS_TSS", "Feature-space WCSS/TSS", "lower is better"),
    ("ARI", "Adjusted Rand index", "higher is better"),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5.7), constrained_layout=True)
for axis, (metric, ylabel, direction) in zip(axes, metrics):
    for method in STABILITY_METHOD_ORDER:
        method_rows = stability_results.loc[stability_results["method"] == method]
        color, marker, linestyle = method_styles[method]
        summary = method_rows.groupby("K")[metric].agg(["median", "min", "max"])
        is_primary = method == "SC-PaCMAP (full graph)"

        axis.fill_between(
            summary.index, summary["min"], summary["max"],
            color=color, alpha=0.16 if is_primary else 0.07, linewidth=0,
        )
        axis.plot(
            summary.index, summary["median"], color=color, marker=marker,
            linestyle=linestyle, linewidth=2.8 if is_primary else 1.5,
            markersize=8 if is_primary else 5, label=method,
        )

    axis.axvline(KNOWN_CLASS_COUNT, color="#333333", linestyle=":", linewidth=1.2)
    axis.set(
        xlabel="Number of clusters K", ylabel=f"{ylabel} ({direction})",
        xticks=list(K_VALUES),
    )
    axis.grid(alpha=0.2)

fig.suptitle("CMiO all-K algorithmic stability on one fixed dataset", fontsize=14)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels, loc="outside lower center", ncol=3,
    title="Lines: seed medians; bands: min-max across 3 seeds",
)
plt.show()

### Reference-K stability cloud

Each faint point is one algorithm-seed run at the label-informed reference $K=7$; the larger outlined marker is the across-seed median. The cloud shows how much optimizer randomness moves the spatial-feature trade-off at the reference $K$, without a formal significance claim.

In [ ]:
stability_results = pd.read_csv(ROOT / "results" / "cmio_algorithm_stability.csv")
method_styles = {
    "UMAP + XYZ": ("#D55E00", "o"),
    "PaCMAP + XYZ": ("#56B4E9", "^"),
    "LocalMAP + XYZ": ("#009E73", "v"),
    "PaCMAP feature-only": ("#7A5195", "P"),
    "SC-PaCMAP (neighbour replacement graph)": ("#E15759", "h"),
    "SC-PaCMAP (full graph)": ("#E60049", "*"),
    "SpaSNE": ("#D4A900", "X"),
}
reference_k = KNOWN_CLASS_COUNT
cloud_results = stability_results.loc[stability_results["K"] == reference_k]

fig, axis = plt.subplots(figsize=(9, 6.5), constrained_layout=True)
for method, (color, marker) in method_styles.items():
    method_rows = cloud_results.loc[cloud_results["method"] == method]
    is_primary = method == "SC-PaCMAP (full graph)"

    axis.scatter(
        method_rows["H_per_sample"], method_rows["WCSS_TSS"],
        s=70 if is_primary else 40, marker=marker, color=color,
        alpha=0.55, linewidth=0, zorder=2,
    )
    axis.scatter(
        method_rows["H_per_sample"].median(), method_rows["WCSS_TSS"].median(),
        s=260 if is_primary else 160, marker=marker, color=color,
        edgecolor="black", linewidth=1.4, label=method, zorder=3,
    )

axis.set(
    xlabel="Mean windowed entropy H/N (lower is better)",
    ylabel="Feature-space WCSS/TSS (lower is better)",
    title=f"CMiO reference-K stability cloud (K={reference_k})",
)
axis.grid(alpha=0.2)
axis.legend(
    title="Faint: seed runs; outlined: seed median",
    bbox_to_anchor=(1.02, 1), loc="upper left",
)
plt.show()

### All-K trade-off range

Marker size increases with $K$. Each faint line is one algorithm seed's path across K, and the bold line with markers traces the across-seed median. This keeps the joint spatial-feature trend readable instead of drawing many equally-dark seed lines.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd


def find_results_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "results" / "cmio_algorithm_stability.csv").is_file():
            return candidate
        nested = candidate / "sc-pacmap-reproducibility"
        if (nested / "results" / "cmio_algorithm_stability.csv").is_file():
            return nested
    raise FileNotFoundError("Could not locate cmio_algorithm_stability.csv")


stability_results = pd.read_csv(
    find_results_root() / "results" / "cmio_algorithm_stability.csv"
)
method_styles = {
    "UMAP + XYZ": ("#D55E00", "o", "-"),
    "PaCMAP + XYZ": ("#56B4E9", "^", "-"),
    "LocalMAP + XYZ": ("#009E73", "v", "-"),
    "PaCMAP feature-only": ("#7A5195", "P", "-"),
    "SC-PaCMAP (neighbour replacement graph)": ("#E15759", "h", "-"),
    "SC-PaCMAP (full graph)": ("#E60049", "*", "-"),
    "SpaSNE": ("#D4A900", "X", "-"),
}
method_order = list(method_styles)
reference_k = 7

fig, axis = plt.subplots(figsize=(11.5, 7.5), constrained_layout=True)
for method in method_order:
    rows = stability_results.loc[stability_results["method"] == method]
    color, marker, linestyle = method_styles[method]
    is_primary = method == "SC-PaCMAP (full graph)"

    # One faint colored line per seed, instead of a solid shaded box per K.
    for seed in ALGORITHM_SEEDS:
        seed_rows = rows.loc[rows["algorithm_seed"] == seed].sort_values("K")
        axis.plot(
            seed_rows["H_per_sample"], seed_rows["WCSS_TSS"], color=color,
            linewidth=1.0, alpha=0.28, zorder=1,
        )

    summary = rows.groupby("K")[["H_per_sample", "WCSS_TSS"]].median()
    sizes = 28 + 7 * (summary.index - summary.index.min())
    axis.plot(
        summary["H_per_sample"], summary["WCSS_TSS"], color=color,
        linestyle=linestyle, linewidth=3.0 if is_primary else 1.7,
        zorder=2, label=method,
    )
    axis.scatter(
        summary["H_per_sample"], summary["WCSS_TSS"], s=sizes,
        color=color, marker=marker,
        edgecolor="black" if is_primary else color,
        linewidth=0.9 if is_primary else 0, zorder=3,
    )

    if is_primary:
        for k in sorted({int(summary.index.min()), reference_k, int(summary.index.max())}):
            axis.annotate(
                f"K={k}", (summary.loc[k, "H_per_sample"], summary.loc[k, "WCSS_TSS"]),
                xytext=(6, 6), textcoords="offset points", color=color,
                fontsize=9, fontweight="bold",
            )

axis.set(
    xlabel="Mean windowed entropy H/N (lower is better)",
    ylabel="Feature-space WCSS/TSS (lower is better)",
    title="CMiO all-K spatial-feature trade-off range",
)
axis.grid(alpha=0.2)
axis.legend(
    title="Faint lines: seed runs; bold line: seed median",
    bbox_to_anchor=(1.02, 1), loc="upper left",
)
plt.show()